# 阶段 13 验证：HDMI 分屏输出

配合 `verify.md`。分三步：

1. 开关与模式互不干扰；
2. **抓三帧做数字判定**（这是关键 —— 光靠眼睛看「左右是不是同一区域」不够硬）；
3. 人眼测试（半张纸）。

### 三帧怎么抓

    A = 全屏原始   （分屏关 + mode0 + PARAM=128 直通 = ×1.0）
    B = 全屏处理   （分屏关 + mode0 + PARAM=255 = ×1.99）
    S = 分屏帧     （分屏开 + mode0 + PARAM=255）

期望：**S 的左半 ≈ A 的某一半、S 的右半 ≈ B 的同一半**。
抓这三次之间**画面必须不动**，否则对不上。


In [ ]:
import sys; sys.path.insert(0, "/home/xilinx/jupyter_notebooks/ai_video_2024.2")
from lib import video_setup as vs, exposure
from lib.ai_video_regs import AiVideoRegs

ROOT = "/home/xilinx/jupyter_notebooks/ai_video_2024.2"
ol, iic, dm, vdma0, vdma1, mode = vs.bring_up(ROOT, manual_exposure=exposure.WORKPOINT)
regs = AiVideoRegs(ol.axis_video_proc_0)

print("VERSION = 0x%08X (阶段 %d)" % (regs.version(), regs.version_stage()))  # 期望 0x0001020D / 13
print("has_split =", regs.has_split())                    # 期望 True
regs.set_split(True); print("split =", regs.split())      # 期望 1
regs.set_mode(2);     print("split =", regs.split())      # 仍应为 1（bit7 和 [1:0] 分开）
regs.set_param(200);  print("PARAM =", regs.param())      # 期望 200（阶段 12 的功能还在）
print("四象限 =", regs.quads())                            # 四项不全为 0
regs.set_split(False)
vs.stop_all_channels(vdma0, vdma1)
from lib import video_path as vp
from lib.ai_video_regs import LUMA_SUM, PIXEL_CNT

vp.start_capture(vdma0, mode)                  # ← 关键：启动 VDMA0 的 S2MM
regs.wait_new_frame(); regs.wait_new_frame()   # 等 1~2 帧，读到整帧统计
print("PIXEL_CNT =", regs.rd(PIXEL_CNT))       # 应 ≈ 921600
print("LUMA_SUM  =", regs.rd(LUMA_SUM))        # 应非 0
print("四象限 =", regs.quads())                # 每块几百，且四块不一样
vs.stop_all_channels(vdma0, vdma1)

In [ ]:
import os
import sys

import numpy as np
import matplotlib.pyplot as plt

ROOT = os.path.abspath(os.getcwd())
while ROOT != os.path.dirname(ROOT):
    if os.path.isfile(os.path.join(ROOT, "lib", "video_setup.py")):
        break
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, ROOT)

from lib import video_setup as vs
from lib import video_path as vp
from lib.ai_video_regs import AiVideoRegs

ol, iic, demosaic, vdma0, vdma1, mode = vs.bring_up(ROOT)
vs.stop_all_channels(vdma0, vdma1)
regs = AiVideoRegs(ol.axis_video_proc_0)

print("VERSION   = 0x%08X (阶段 %d)" % (regs.version(), regs.version_stage()))
print("has_split =", regs.has_split())
assert regs.has_split(), "板上不是阶段 13 的 bitstream，先跑 build_overlay.sh 13"

# 先把 HDMI 显示通路开起来 —— 否则显示器是黑的，看不到分屏效果。
# 它内部会 tie(VDMA0) + tie(VDMA1) + start 四个通道，之后抓帧照样能用。
vp.start_hdmi_passthrough(vdma0, vdma1, mode)
regs.set_split(True)       # 立刻打开分屏：一跑完这格，显示器上就是分屏画面
regs.set_mode(0)
regs.set_param(255)        # 增益 x1.99：左右亮度差最明显，最容易看出效果
regs.wait_new_frame()
print("HDMI 显示通路已启动 —— 现在显示器上应该能看到实时画面。")

# 注意：这里【不要】再调 start_capture()。它内部第一句是 vdma0.readchannel.stop()，
# 会把 start_hdmi_passthrough 建立起来的 tie 和 MM2S 一起停掉 —— 显示器立刻变黑。
# start_hdmi_passthrough 已经把采集通道启动好了，后面的 readframe() 抓帧照样能用。
regs.wait_new_frame()
print("采集已启动。请把画面停在一个有明暗纹理的场景上（别用白墙）。")


## 验证 1：开关和模式互不干扰


In [ ]:
regs.set_split(False)
regs.set_mode(2)
print("split=%d mode=%d   (期望 0, 2)" % (regs.split(), regs.mode_sel()))

regs.set_split(True)
print("split=%d mode=%d   (期望 1, 2)" % (regs.split(), regs.mode_sel()))

regs.set_mode(1)          # 切模式不能把分屏冲掉
print("split=%d mode=%d   (期望 1, 1)" % (regs.split(), regs.mode_sel()))

regs.set_split(False)     # 关分屏不能把模式冲掉
print("split=%d mode=%d   (期望 0, 1)" % (regs.split(), regs.mode_sel()))

assert regs.split() == 0 and regs.mode_sel() == 1, "开关和模式互相干扰了！"
print("验证 1 通过")


## 验证 2：抓三帧 + 数字判定


In [ ]:
def grab(mode_sel, param, split, warm=5):
    regs.set_split(split)
    regs.set_mode(mode_sel)
    regs.set_param(param)
    for _ in range(warm):
        regs.wait_new_frame()
    return vp.frame_to_rgb(vp.capture_frame(vdma0)).astype(np.float32)


# ===== 图像在这里，三次抓帧之间别动画面 =====
A = grab(0, 128, False)   # 全屏原始（直通）
B = grab(0, 255, False)   # 全屏处理（增益 x1.99）
S = grab(0, 255, True)    # 分屏：左原始 + 右处理

print("A(原始)  均值=%.1f  B(处理)  均值=%.1f  S(分屏)  均值=%.1f"
      % (A.mean(), B.mean(), S.mean()))
print("帧尺寸:", A.shape)

# ---- 存盘（把这三个 PNG 发我，我一眼就能看出问题）--------------------
from PIL import Image
for _n, _im in (("A_raw", A), ("B_proc", B), ("S_split", S)):
    Image.fromarray(_im.astype(np.uint8)).save("/home/xilinx/%s.png" % _n)
print("已存到板上：/home/xilinx/A_raw.png  /home/xilinx/B_proc.png  /home/xilinx/S_split.png")

# ---- 列亮度曲线：一眼看出每一段是什么、黑带在哪 -----------------------
def col_profile(img):
    return img.reshape(img.shape[0], -1, 64, 3).mean(axis=(0, 2, 3))


pS, pA, pB = col_profile(S), col_profile(A), col_profile(B)
print()
print("每一格 = 64 列的平均亮度（共 20 格，正好覆盖 1280 列）：")
print("  格号    " + " ".join("%4d" % i for i in range(len(pS))))
print("  S(分屏) " + " ".join("%4d" % v for v in pS))
print("  A(原始) " + " ".join("%4d" % v for v in pA))
print("  B(处理) " + " ".join("%4d" % v for v in pB))
print()
print("怎么看：S 的前 10 格应该跟 A 的某 10 格接近、后 10 格跟 B 的同一段接近；")
print("        出现连续几格接近 0 的就是那条不该有的黑带。")


In [ ]:
h, w = S.shape[:2]
half = w // 2
L = S[:, :half]      # 分屏帧的左半
R = S[:, half:]      # 分屏帧的右半

fig, ax = plt.subplots(1, 4, figsize=(24, 4.6))
ax[0].imshow(A.astype(np.uint8)); ax[0].set_title("A: full frame, split OFF, PARAM=128 (raw)")
ax[1].imshow(B.astype(np.uint8)); ax[1].set_title("B: full frame, split OFF, PARAM=255 (processed)")
ax[2].imshow(S.astype(np.uint8)); ax[2].set_title("S: split ON  (left=raw, right=processed)")
ax[3].imshow(S.astype(np.uint8)); ax[3].axvline(half - 0.5, color="r", lw=2)
ax[3].set_title("S with the x=%d boundary marked" % half)
for a in ax:
    a.axis("off")
plt.tight_layout(); plt.show()


### 数字判定：S 的左右两半分别对应哪一半

HDL 的分屏规则是：**显示器左半 = 原始、右半 = 处理后的同一区域**。

但 `frame_to_rgb()` 会把画面水平镜像（摄像头本身的镜像），所以数组里
**左半其实对应显示器右半**。用同一次采集的三个帧互相算 RMS 就能把它定死：

* 数组左半 ↔ `B` 的**右半**（处理后，同一区域）
* 数组右半 ↔ `A` 的**右半**（原始，同一区域）

RMS < 10 视为匹配；> 30 视为不匹配（实测约 3.9 / 83）。


In [ ]:
def rms(a, b):
    return float(np.sqrt(((a.astype(np.float32) - b.astype(np.float32)) ** 2).mean()))


# 配对说明（最早这里配反过，导致「明明成功却报没通过」）：
#   HDL 分屏：显示器【左半】= 原始、右半 = 处理后，两边是同一段源列（1:1 不缩放）。
#   而 frame_to_rgb() 又做了一次水平镜像 => 数组的【左半】其实对应【显示器右半】。
#   所以正确配对是：
#       数组左半 <-> B 的右半（处理后，同一区域）
#       数组右半 <-> A 的右半（原始，  同一区域）
cand = {
    "S_left  vs B_right (processed, SAME region)":  (rms(S[:, :half], B[:, half:]), "match"),
    "S_right vs A_right (raw,       SAME region)":  (rms(S[:, half:], A[:, half:]), "match"),
    "S_left  vs B_left  (processed, OTHER region)": (rms(S[:, :half], B[:, :half]), "diff"),
    "S_left  vs A_left  (raw,       OTHER region)": (rms(S[:, :half], A[:, :half]), "diff"),
    "S_right vs B_right (processed, WRONG half)":   (rms(S[:, half:], B[:, half:]), "diff"),
}
for k, (v, _kind) in cand.items():
    print("  %-46s RMS = %6.2f" % (k, v))

m_proc = cand["S_left  vs B_right (processed, SAME region)"][0]
m_raw  = cand["S_right vs A_right (raw,       SAME region)"][0]
print()
print("同一区域匹配项：数组左半 = 处理后 RMS = %.2f   数组右半 = 原始 RMS = %.2f" % (m_proc, m_raw))
print()
# ---- 判据修正（2026-10-14）：绝对阈值会随画面纹理强弱变化 ----------------
# 上面那几个 RMS 都是**跨帧**比较（S 和 A/B 不是同一帧），同样一点跨帧噪声，
# 在纹理丰富的画面上能放大到 25，在平画面上只有 4（10-08 那次就是 3.9）。
# 所以除了绝对阈值，再加两条稳的：
#   1) 相对判据：匹配项至少比最小错配项小 3 倍；
#   2) **同帧自洽**（最强）：分屏帧里左半 = 处理后、右半 = 原始，而且是**同一段源列**；
#      mode 0 的增益是确定性的 out = clip((in * PARAM) >> 7)，所以左半应该
#      逐像素等于「右半 × 255/128 饱和后的结果」—— 同一帧内比较，不含跨帧噪声，
#      RMS 应该 < 3（只剩整数舍入）。这一条能直接证明「左右是同一区域的原始/处理后」。
import numpy as _np
_exp = _np.clip((S[:, half:].astype(_np.int32) * 255) >> 7, 0, 255)   # 抓帧时 PARAM=255
rms_same = rms(S[:, :half], _exp)
# 主判据 2：列剖面吻合。每格 = 64 列 × 720 行的均值，跨帧噪声会被平均掉，
#           所以这个比较比逐像素 RMS 稳得多（用中位数判，避开过渡段的单格抖动）。
_d1 = _np.abs(pS[:10] - pB[10:])      # 数组左半(处理后)  vs  B 的右半
_d2 = _np.abs(pS[10:] - pA[10:])      # 数组右半(原始)    vs  A 的右半
# 旁证："不同区域"的错配项必须远大于匹配项 —— 证明左右不是各取画面的一半。
# 注意 S_right vs B_right（同区域 raw vs processed）**不是错配**，它反映的是处理增益，
# 别把它算进错配项里（2026-10-14 这次就是栽在这里：它的差异本来就小，
# 暗处 4→7、亮处 174→235，RMS 只有 88 -> 相对判据被它拉低到 1.8 倍）。
_other = min(rms(S[:, :half], B[:, :half]), rms(S[:, :half], A[:, :half]))
print()
print("同帧自洽：S_left vs clip(S_right * 255/128)   RMS = %.2f   （主判据，要求 < 3）" % rms_same)
print("列剖面吻合：前10格 vs B后10格  中位差 %.1f  最大差 %.0f"
      % (_np.median(_d1), _d1.max()))
print("            后10格 vs A后10格  中位差 %.1f  最大差 %.0f"
      % (_np.median(_d2), _d2.max()))
print("不同区域错配项（应 >100，旁证左右不是各取一半）：%.1f" % _other)
print()
_prof_ok = (_np.median(_d1) < 5 and _np.median(_d2) < 5
            and _d1.max() < 20 and _d2.max() < 20)
if rms_same < 3.0 and _prof_ok and _other > 100:
    print("验证 2 通过：左右两边是【同一个画面区域】的「原始」与「处理后」")
    print("  -> 显示器上：左半 = 原始、右半 = 处理后的同一区域")
    print("  -> 旁证：S_left 对 B_left 的 RMS 很大，正说明两边不是各取画面的一半")
else:
    print("验证 2 没通过 —— 把上面几行 RMS 发我")


## 验证 3：人眼测试（最直观，答辩可以现场做）

拿一张写着字的纸，**只让左边那半个字入画**（右半边对着一块平整的东西）：

* 屏幕**左右两半都应该显示这半个字**（一边原始、一边处理过）；
* 如果左边显示左边内容、右边显示右边内容，那说明设计不对。

也可以切 `MODE_SEL=0 / PARAM=255`（增益）：左半应该明显比右半暗。


In [ ]:
# 方便现场切换：
def show(mode_sel, param, split=True):
    regs.set_split(split)
    regs.set_mode(mode_sel)
    regs.set_param(param)
    print("已设置 mode=%d PARAM=%d split=%d" % (mode_sel, param, split))


show(0, 255)   # 增益 x1.99 —— 左右亮度差最明显
# show(2, 255) # 锐化最强 —— 右半更脆
# show(1, 255) # 逆光补偿最强 —— 右半更平
# show(0, 192, split=False)   # 关分屏，回到阶段 12 的样子


## 收尾


In [ ]:
regs.set_split(False)
regs.set_mode(0)
regs.set_param(192)
vs.stop_all_channels(vdma0, vdma1)
print("[ok] 分屏已关、模式恢复 0 / PARAM=192、VDMA 通道已停")
print("注意：这一格停掉了 VDMA，HDMI 会黑屏（这是设计好的收尾动作）。")
print("      想继续看分屏，往下跑「恢复显示」那一格即可（不用重启内核）。")
try:
    ol.free()
    print("[ok] overlay 已释放")
except Exception as e:
    print("[warn] %s" % e)


## 收尾之后想再看分屏：恢复显示

上面那一格（收尾）停掉了 VDMA 并释放了 overlay，所以 HDMI 黑屏 —— **这是设计好的行为，不是哪里点错了**。

**任何时候跑下面这一格，画面就会被重新点起来**：

* 刚跑过收尾（overlay 已释放）→ 它会重新下载 bitstream；
* 没跑收尾 → 它只是把四个 VDMA 通道重新启动一遍。

演示的时候：先跑这一格，再用上面 `show(0, 255)` 那格对着镜头调参数。


In [ ]:
# 恢复显示 —— 黑屏时跑这一格（收尾格之后也能救回来）
import os
import sys

ROOT = os.path.abspath(os.getcwd())
while ROOT != os.path.dirname(ROOT):
    if os.path.isfile(os.path.join(ROOT, "lib", "video_setup.py")):
        break
    ROOT = os.path.dirname(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from lib import video_setup as vs
from lib import video_path as vp
from lib.ai_video_regs import AiVideoRegs

try:
    vs.stop_all_channels(vdma0, vdma1)   # overlay 已被 free 时旧对象会失效，忽略即可
except Exception:
    pass

# 重新下载 overlay（幂等：重复跑没问题）
ol, iic, demosaic, vdma0, vdma1, mode = vs.bring_up(ROOT)
regs = AiVideoRegs(ol.axis_video_proc_0)
print("VERSION = 0x%08X (阶段 %d)" % (regs.version(), regs.version_stage()))

vp.start_hdmi_passthrough(vdma0, vdma1, mode)
regs.wait_new_frame()

# 分屏打开 + 增益拉满，左右亮度差最直观
regs.set_split(True)
regs.set_mode(0)
regs.set_param(255)
print("split=%d mode=%d PARAM=%d" % (regs.split(), regs.mode_sel(), regs.param()))
print("显示器上现在应该是：左半 = 原始，右半 = 处理后（右半明显更亮）")
